In [ ]:
# input text that the model will be trained on
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [ ]:
# all unique characters
chars = sorted((list(set(text))))
vocab_size = len(chars)

In [ ]:
# create mapping form characters to integers
# Other mappings: google uses sentencepiece, openai uses tiktoken
# Balance between length of integers with length of vocab
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s]   ## take string, output list of integers
decode = lambda l: ''.join([itos[i] for i in l])  ## take list of integers, output a string

In [ ]:
# now can encode dataset and into a tensor
import torch
data = torch.tensor(encode(text), dtype=torch.long)

In [ ]:
# split into train and validation sets
# helps understand whether its overfitting
n = int(0.9*len(data))
train_data = data[:n]  ## first 90 percent
val_data = data[n:]

In [ ]:
# size of input
block_size = 8
train_data[:block_size+1]

In [ ]:
# reducing block size means efficienct and for model to be familiar with any block size input up to "block_size"
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"when input is {context} that target: {target}")

In [ ]:
# mini batches of blocks since GPUs are very good as parallel processing
torch.manual_seed(1337)
batch_size = 4  ## how many independent sequences will be processed in parallel
block_size = 8  ## what is the maximum context length for predictions

def get_batch(split):
    # generate a small batch
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))  ## random integers of size batch_size
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

print('----')

for b in range(batch_size):  ## batch dimension
    for t in range(block_size):  ## time dimension
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"when input is {context.tolist()} the target: {target}")

In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    
    def __init__(self, vocab_size):
        super().__init__()
        # a lookup table of shape vocab_size*vocab_size
        # each token id maps to a vector os size vocab_size
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)
    
    def forward(self, idx, targets=None):
        # each token id in the (batch_size, block_size) tensor is replaced by a new dimension:
        # list of integers of length vocab_size representing the embedding
        logits = self.token_embedding_table(idx)
        
        if targets is None:
            loss = None
        else:
            # reshape to work with cross_entropy function expectations
            B, T, C = logits.shape  ## batch_size, block_size, vocab size
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)  ## computes loss between predictions and targets

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (batch size by token size) array of indices
        # For each batch, u'd get the new tokens; generates predictions for all batches at once
        for _ in range(max_new_tokens):
            logits, loss = self(idx)  ## get predictions for all positions
            logits = logits[:, -1, :]  ## keep only last postiion's predictions (next token) for each batch
            probs = F.softmax(logits, dim=-1)  ## make em sum to 1 to get probabilities
            idx_next = torch.multinomial(probs, num_samples=1)  ## randomly sample on token from distribution
            idx = torch.cat((idx, idx_next), dim=1)  ## append sampled index to the running sequence
        return idx

m = BigramLanguageModel(vocab_size)
out = m(xb, yb)

## Sample generation
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=100)[0].tolist()))

In [ ]:
# create a pytorch optimiser (an algorithm that updates model weights to reduce loss)
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

In [ ]:
batch_size = 32
for steps in range(1000):
    # sample a batch of data
    xb, yb = get_batch('train')
    
    # evaluate the loss
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backwards()
    optimizer.step()
    
    print(loss.item())

## The mathematical trick in self-attention

In [ ]:
# super small example

torch.manual_seed(1337)
B,T,C = 4,8,2  ## batch, time, channel
x = torch.rand(B,T,C)
x.shape

# v1: For each batch and each postion, computer the average of all tokens up to that position
# bow: bag of words
xbow = torch.zeros((B,T,C))  ## xbow[b,t] is the running average of everything seen so far
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] ## (t+1, C) for one batch
        xbow[b,t] = torch.mean(xprev, 0)  ## each position of embedding C is averaged separately

# v2: Lets use matric multiplication
wei = torch.tril(torch.ones(T, T))
wei = wei / wei.sum(1, keepdim=True)
xbow2 = wei @ x  ## (B, T, T) @ (B, T, C) -> (B, T, C)
torch.allclose(xbow, xbow2)  ## They are the same

# v3: Using softmax
tril = torch.tril(torch.ones(T, T))
wei = torch.zeros((T, T))  ## Make everything 0
wei = wei.masked_fill(tril == 0, float('-inf'))  ## For elements where tril ==0, make neg infinity
wei = F.softmax(wei, dim=-1)
xbow3 = wei @ x
torch.allclose(xbow, xbow3)  ## They are the same

In [ ]:
# v4: self-attention
# Want to information from past tokens in a data-informative way, not just average like before
# Every token will emit a query (what am i looking for) and a key (what do I contain)
torch.manual_seed(1337)
B,T,C = 4,8,32  ## batch, time, channels
x = torch.randn(B,T,C)

# single Head performing self-attention
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x)  ## (B, T, 16)
q = query(x)   ## (B, T, 16)
# all queries dot product all keys
wei = q @ k.transpose(-2, -1) * head_size**-0.5  ## (B, T, 16) @ (B, 16, T) ---> (B, T, T)

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))  ## For elements where tril ==0, make neg infinity
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v

## Notes:
- Attention is a communication mechanism
- Batches are separate
- Alternative option: encoder attention block allows all tokens to interact with each other - often used for sentiment analysis
- Self-attention: keys and values are produced from the same source as queries
- Cross-attention: queries still get produced form x, but the keys and values come from some other, external source (e.g. encoder module)
- Dividing by the root head size means the attention distributed across multiple tokens and isn't squashed

In [ ]:
## Normalises rows
class BatchNormId:
    
    def __init__(self, dim, eps=1e-5, momentum=0.1):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeroes(dim)
    
    def __call__(self, x):
        # calculate the forward pass
        xmean = x.mean(1, keepdim=True)
        xvar = x.var(1, keepdim=True)
        xhat = (x - xmean) / torch.sqrt(xvar + self.eps)
        self.out = self.gamma * xhat + self.beta
        return self.out

    def parameters(self):
        return [self.gamma, self.beta]

torch.manual_seed(1337)
module = BatchNormId(100)
x = torch.randn(32, 100)  ## batch size 32 of 100-dimensional vectors
x = module(x)
x.shape